<a href="https://colab.research.google.com/github/diazcas2/mIArmario/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install google-search-results --quiet
!pip install google-genai --quiet
!pip install torch transformers google-generativeai langdetect --quiet

In [2]:
import json
import re
import time
import torch
import google.generativeai as genai
from transformers import pipeline
from google.colab import userdata
from serpapi import GoogleSearch
import requests

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/Colab Notebooks/MIARMARIO/')

from Modulo1 import *
from Modulo3 import *


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
whisper = pipeline(
  "automatic-speech-recognition",
  "openai/whisper-large-v3",
  dtype=_dtype
)

In [4]:
GEMINI_API_KEY=userdata.get('GEMINI_API_KEY')
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')   # Google Cloud → Custom Search API
SEARCH_ENGINE_ID = userdata.get('SEARCH_ENGINE_ID') # programmablesearchengine.google.com → cx

SERPAPI_KEY = userdata.get('SERPAPI_KEY')  # serpapi.com → dashboard → API Key

genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel("gemini-2.5-flash")

In [5]:
resultado_mod1 = modulo1_entrada(
    fuente="texto",
    texto="hola quiero que me digas un outfit que sea bueno para una cena esta noche, vienen mi jefe a si que tengo que ir arreglado",
    ciudad="Sevilla",
    model=model
)



In [6]:
print(json.dumps(resultado_mod1, ensure_ascii=False, indent=2))

{
  "texto": "Quiero un outfit para una cena esta noche. Mi jefe viene, así que necesito ir arreglado.",
  "fuente": "texto",
  "idioma": "es",
  "contexto": {
    "ciudad": "Sevilla",
    "temperatura": 19.2,
    "unidad": "celsius"
  }
}


In [7]:
armario = {
    "armario": [
        {
            "id": "prenda_001",
            "tipo": "camisa",
            "color": "blanco",
            "estampado": "liso",
            "material": "algodón",
            "formalidad": "formal",
            "temporada": ["primavera", "verano", "otoño"],
            "imagen_path": "fotos/camisa_blanca_001.jpg"
        },
        {
            "id": "prenda_002",
            "tipo": "pantalón",
            "color": "negro",
            "estampado": "liso",
            "material": "lana",
            "formalidad": "formal",
            "temporada": ["otoño", "invierno"],
            "imagen_path": "fotos/pantalon_negro_002.jpg"
        }
    ]
}

resultado_mod3 = modulo3_seleccion_outfit(
    armario_json=armario,
    instruccion_usuario=resultado_mod1["texto"],
    contexto={"temperatura": 18, "ocasion": "cena"},
    model=model,
    SERPAPI_KEY=SERPAPI_KEY
)



[Módulo 3] Queries generadas: ['zapatos negros formales hombre zara', 'cinturón cuero negro formal mango', 'corbata azul marino formal zalando']
{'search_metadata': {'id': '69b7ee6bbadf0b018a2cf70d', 'status': 'Success', 'json_endpoint': 'https://serpapi.com/searches/bHA2zJr6vNMknxZbDivDkg/69b7ee6bbadf0b018a2cf70d.json', 'pixel_position_endpoint': 'https://serpapi.com/searches/bHA2zJr6vNMknxZbDivDkg/69b7ee6bbadf0b018a2cf70d.json_with_pixel_position', 'created_at': '2026-03-16 11:50:03 UTC', 'processed_at': '2026-03-16 11:50:03 UTC', 'google_url': 'https://www.google.com/search?q=zapatos+negros+formales+hombre+zara+comprar&oq=zapatos+negros+formales+hombre+zara+comprar&hl=es&gl=es&num=5&sourceid=chrome&ie=UTF-8', 'raw_html_file': 'https://serpapi.com/searches/bHA2zJr6vNMknxZbDivDkg/69b7ee6bbadf0b018a2cf70d.html', 'total_time_taken': 0.69}, 'search_parameters': {'engine': 'google', 'q': 'zapatos negros formales hombre zara comprar', 'google_domain': 'google.com', 'hl': 'es', 'gl': 'es', 

In [8]:
print(json.dumps(resultado_mod3, ensure_ascii=False, indent=2))

{
  "outfit": {
    "prendas": [
      {
        "id": "prenda_001",
        "tipo": "camisa",
        "color": "blanco"
      },
      {
        "id": "prenda_002",
        "tipo": "pantalón",
        "color": "negro"
      },
      {
        "id": "zapatos_negros_formales",
        "tipo": "zapatos",
        "color": "negro"
      },
      {
        "id": "cinturon_negro",
        "tipo": "cinturon",
        "color": "negro"
      },
      {
        "id": "corbata_azul",
        "tipo": "corbata",
        "color": "azul"
      }
    ],
    "estilo": "formal",
    "ocasion": "cena",
    "temperatura": 18
  },
  "prompt_imagen": "A stylish man wearing a crisp white cotton formal shirt, tailored black wool formal pants, black leather dress shoes with laces, a black leather belt, and a navy blue formal tie. The man is standing, posing for a formal dinner with his boss. Full body, fashion photography, studio lighting",
  "referencias_tiendas": [
    {
      "prenda": "Zapatos de Vestir Ho